# HMM Market Regime Detection - Exploratory Data Analysis

This notebook demonstrates the complete workflow for Hidden Markov Model-based market regime detection.

## Overview
- Load and explore OHLC data
- Compute technical features
- Train HMM models
- Visualize detected regimes
- Analyze regime characteristics
- Demonstrate inference capabilities

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import our HMM system modules
import sys
sys.path.append('../src')

from features import compute_features, validate_features
from model_wrapper import HMMRegimeModel, select_best_model
from utils import (
    setup_logging, set_random_seeds, label_states_by_return_volatility,
    calculate_performance_metrics
)
from rolling_train import RollingHMMTrainer
from inference import HMMInferenceEngine
from backtest import HMMBacktester, SimpleRegimeStrategy

# Set up plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_palette("husl")

# Set random seeds for reproducibility
set_random_seeds(42)

print("HMM Market Regime Detection System - Ready for Analysis")

## 1. Data Loading and Exploration

Let's start by generating sample market data and exploring its characteristics.

In [ ]:
# Generate realistic market data with regime changes
np.random.seed(42)

def generate_market_data(n_days=1000, initial_price=100):
    """Generate synthetic market data with different regime characteristics."""
    
    dates = pd.date_range('2020-01-01', periods=n_days, freq='D')
    
    # Define three regimes with different characteristics
    regimes = []
    returns = []
    
    # Regime transitions (roughly every 100-200 days)
    regime_changes = [0, 200, 400, 650, 850, n_days]
    regime_types = ['Bull', 'Sideways', 'Bear', 'Bull', 'Sideways']
    
    for i in range(len(regime_changes)-1):
        start_idx = regime_changes[i]
        end_idx = regime_changes[i+1]
        period_length = end_idx - start_idx
        regime_type = regime_types[i]
        
        if regime_type == 'Bull':
            # Bull market: positive drift, moderate volatility
            period_returns = np.random.normal(0.0008, 0.012, period_length)
        elif regime_type == 'Bear':
            # Bear market: negative drift, high volatility
            period_returns = np.random.normal(-0.0005, 0.018, period_length)
        else:  # Sideways
            # Sideways market: no drift, low volatility
            period_returns = np.random.normal(0.0001, 0.008, period_length)
        
        returns.extend(period_returns)
        regimes.extend([regime_type] * period_length)
    
    # Generate prices
    returns = np.array(returns)
    prices = initial_price * (1 + returns).cumprod()
    
    # Generate OHLC data
    noise_scale = 0.003
    ohlc_data = pd.DataFrame({
        'Open': prices + np.random.normal(0, noise_scale, n_days),
        'Close': prices,
        'Volume': np.random.lognormal(10, 0.5, n_days)
    }, index=dates)
    
    # Generate High and Low based on intraday volatility
    intraday_range = np.abs(np.random.normal(0, 0.01, n_days))
    ohlc_data['High'] = ohlc_data[['Open', 'Close']].max(axis=1) + intraday_range
    ohlc_data['Low'] = ohlc_data[['Open', 'Close']].min(axis=1) - intraday_range
    
    # True regimes for comparison
    true_regimes = pd.Series(regimes, index=dates, name='True_Regime')
    
    return ohlc_data, true_regimes

# Generate sample data
ohlc_data, true_regimes = generate_market_data(1000)

print(f"Generated {len(ohlc_data)} days of market data")
print(f"Price range: {ohlc_data['Close'].min():.2f} - {ohlc_data['Close'].max():.2f}")
print(f"True regime distribution:\n{true_regimes.value_counts()}")

ohlc_data.head()

In [ ]:
# Plot price data with true regimes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Price chart
ax1.plot(ohlc_data.index, ohlc_data['Close'], linewidth=1.5, alpha=0.8)
ax1.set_title('Market Data with True Regimes', fontsize=16, fontweight='bold')
ax1.set_ylabel('Price', fontsize=12)
ax1.grid(True, alpha=0.3)

# Color background by regime
regime_colors = {'Bull': 'green', 'Bear': 'red', 'Sideways': 'gray'}
for regime in regime_colors:
    mask = true_regimes == regime
    if mask.any():
        ax1.fill_between(ohlc_data.index, ohlc_data['Close'].min(), ohlc_data['Close'].max(),
                        where=mask, alpha=0.1, color=regime_colors[regime], label=f'{regime} Regime')

ax1.legend(loc='upper left')

# Daily returns
daily_returns = ohlc_data['Close'].pct_change().dropna()
ax2.plot(daily_returns.index, daily_returns, alpha=0.6, linewidth=0.8)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('Daily Returns', fontsize=14)
ax2.set_ylabel('Return', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics by regime
print("\nReturn Statistics by True Regime:")
for regime in ['Bull', 'Bear', 'Sideways']:
    regime_mask = true_regimes == regime
    regime_returns = daily_returns[regime_mask[1:]]  # Align with returns
    if len(regime_returns) > 0:
        print(f"{regime:>8}: Mean={regime_returns.mean():+.4f}, Std={regime_returns.std():.4f}, "
              f"Sharpe={regime_returns.mean()/regime_returns.std()*np.sqrt(252):.2f}")

## 2. Feature Engineering

Now let's compute the technical features used for HMM training.

In [ ]:
# Compute technical features
features = compute_features(ohlc_data)

# Validate features
is_valid, validation_msg = validate_features(features)
print(f"Feature validation: {validation_msg}")

print(f"\nComputed features shape: {features.shape}")
print(f"Feature columns: {list(features.columns)}")
print(f"Date range: {features.index.min()} to {features.index.max()}")

# Feature statistics
print("\nFeature Statistics:")
features.describe()

In [ ]:
# Plot features
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

feature_names = ['log_return', 'volatility', 'RSI', 'MACD', 'BBW']
feature_titles = ['Log Returns', 'Rolling Volatility (20d)', 'RSI (14d)', 'MACD Histogram', 'Bollinger Band Width']

for i, (feature, title) in enumerate(zip(feature_names, feature_titles)):
    ax = axes[i]
    ax.plot(features.index, features[feature], alpha=0.8, linewidth=1)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add regime background coloring
    for regime in regime_colors:
        mask = true_regimes[features.index] == regime
        if mask.any():
            y_min, y_max = ax.get_ylim()
            ax.fill_between(features.index, y_min, y_max,
                          where=mask, alpha=0.1, color=regime_colors[regime])

# Remove empty subplot
axes[-1].remove()

plt.tight_layout()
plt.show()

# Feature correlations
plt.figure(figsize=(10, 8))
correlation_matrix = features.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
           square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. HMM Model Training and Selection

Let's train HMM models and select the best configuration.

In [ ]:
# Prepare training data
from sklearn.preprocessing import StandardScaler

# Use first 80% for training
train_size = int(len(features) * 0.8)
train_features = features.iloc[:train_size].copy()
test_features = features.iloc[train_size:].copy()

print(f"Training data: {len(train_features)} samples")
print(f"Test data: {len(test_features)} samples")

# Scale features
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features.values)
test_scaled = scaler.transform(test_features.values)

print(f"Scaled training data shape: {train_scaled.shape}")

# Model selection across different number of states
best_model, selection_results = select_best_model(
    train_scaled,
    n_states_range=[2, 3, 4, 5],
    n_runs=3,
    max_iter=150
)

print(f"\nBest model: {best_model.n_states} states")
print(f"Best BIC: {selection_results['best_bic']:.2f}")

# Show selection results
results_df = pd.DataFrame(selection_results['all_results'])
print("\nModel Selection Results:")
summary = results_df.groupby('n_states').agg({
    'log_likelihood': ['mean', 'std'],
    'bic': ['mean', 'std'],
    'converged': 'mean'
}).round(4)

print(summary)

In [ ]:
# Analyze the best model
model_info = best_model.get_model_info()

print("Best Model Analysis")
print("="*50)
print(f"Number of states: {model_info['n_states']}")
print(f"Converged: {model_info['converged']}")
print(f"Log-likelihood: {model_info['log_likelihood']:.4f}")

# Transition matrix
A = model_info['transition_matrix']
print("\nTransition Matrix:")
transition_df = pd.DataFrame(A, 
                           index=[f'State_{i}' for i in range(len(A))],
                           columns=[f'State_{i}' for i in range(len(A))])
print(transition_df.round(3))

# Visualize transition matrix
plt.figure(figsize=(8, 6))
sns.heatmap(A, annot=True, cmap='Blues', square=True, 
           xticklabels=[f'State {i}' for i in range(len(A))],
           yticklabels=[f'State {i}' for i in range(len(A))])
plt.title('HMM Transition Matrix', fontweight='bold')
plt.ylabel('From State')
plt.xlabel('To State')
plt.tight_layout()
plt.show()

# Emission parameters
emission_params = model_info['emission_params']
print("\nEmission Parameters (Means):")
means_df = pd.DataFrame(emission_params['means'], 
                       columns=train_features.columns,
                       index=[f'State_{i}' for i in range(len(emission_params['means']))])
print(means_df.round(4))

## 4. Regime Detection and Analysis

Let's use the trained model to detect regimes and analyze the results.

In [ ]:
# Predict regimes on full dataset
full_scaled = scaler.transform(features.values)

# Viterbi decoding (most likely path)
predicted_states = best_model.predict(full_scaled)
posterior_probs = best_model.predict_proba(full_scaled)

# Label states based on return characteristics
state_labels = label_states_by_return_volatility(
    predicted_states, 
    features['log_return']
)

print("State Labels:")
for state_idx, label in state_labels.items():
    print(f"State {state_idx}: {label}")

# Map states to regime labels
predicted_regimes = pd.Series(
    [state_labels[state] for state in predicted_states],
    index=features.index,
    name='Predicted_Regime'
)

print(f"\nPredicted regime distribution:")
print(predicted_regimes.value_counts())

# Compare with true regimes
aligned_true_regimes = true_regimes[features.index]
comparison_df = pd.DataFrame({
    'True': aligned_true_regimes,
    'Predicted': predicted_regimes
})

print("\nRegime Classification Accuracy:")
accuracy = (comparison_df['True'] == comparison_df['Predicted']).mean()
print(f"Overall accuracy: {accuracy:.3f}")

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(comparison_df['True'], comparison_df['Predicted'])
cm_df = pd.DataFrame(cm, 
                    index=['True_' + r for r in ['Bear', 'Bull', 'Sideways']], 
                    columns=['Pred_' + r for r in ['Bear', 'Bull', 'Sideways']])
print("\nConfusion Matrix:")
print(cm_df)

In [ ]:
# Plot detected regimes vs true regimes
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

# Price with true regimes
ax = axes[0]
ax.plot(ohlc_data.index, ohlc_data['Close'], 'k-', linewidth=1, alpha=0.8)
for regime in regime_colors:
    mask = true_regimes == regime
    if mask.any():
        ax.fill_between(ohlc_data.index, ohlc_data['Close'].min(), ohlc_data['Close'].max(),
                       where=mask, alpha=0.2, color=regime_colors[regime], label=f'True {regime}')
ax.set_title('True Regimes', fontweight='bold')
ax.set_ylabel('Price')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Price with predicted regimes
ax = axes[1]
ax.plot(ohlc_data.index, ohlc_data['Close'], 'k-', linewidth=1, alpha=0.8)
for regime in regime_colors:
    mask = predicted_regimes == regime
    if mask.any():
        ax.fill_between(features.index, ohlc_data['Close'].min(), ohlc_data['Close'].max(),
                       where=mask, alpha=0.2, color=regime_colors[regime], label=f'Predicted {regime}')
ax.set_title('Predicted Regimes (HMM)', fontweight='bold')
ax.set_ylabel('Price')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Regime probabilities
ax = axes[2]
for i, (state_idx, label) in enumerate(state_labels.items()):
    ax.plot(features.index, posterior_probs[:, state_idx], 
           label=f'P({label})', alpha=0.8, linewidth=1.5)
ax.set_title('Regime Posterior Probabilities', fontweight='bold')
ax.set_ylabel('Probability')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. Regime Characteristics Analysis

In [ ]:
# Analyze characteristics of each detected regime
regime_stats = {}

for regime in predicted_regimes.unique():
    mask = predicted_regimes == regime
    regime_returns = features.loc[mask, 'log_return']
    regime_vol = features.loc[mask, 'volatility']
    
    stats = {
        'frequency': mask.mean(),
        'avg_duration': calculate_avg_regime_duration(predicted_states[mask.values], 
                                                    list(state_labels.keys())[list(state_labels.values()).index(regime)]),
        'mean_return': regime_returns.mean(),
        'return_std': regime_returns.std(),
        'mean_volatility': regime_vol.mean(),
        'sharpe_ratio': regime_returns.mean() / regime_returns.std() * np.sqrt(252) if regime_returns.std() > 0 else 0,
        'skewness': regime_returns.skew(),
        'kurtosis': regime_returns.kurtosis()
    }
    
    regime_stats[regime] = stats

def calculate_avg_regime_duration(states, target_state):
    """Calculate average duration in a regime."""
    durations = []
    current_duration = 0
    
    for state in states:
        if state == target_state:
            current_duration += 1
        else:
            if current_duration > 0:
                durations.append(current_duration)
                current_duration = 0
    
    if current_duration > 0:
        durations.append(current_duration)
    
    return np.mean(durations) if durations else 0

# Display regime statistics
regime_stats_df = pd.DataFrame(regime_stats).T
print("Regime Characteristics:")
print("=" * 80)
print(regime_stats_df.round(4))

# Plot regime return distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, regime in enumerate(predicted_regimes.unique()):
    mask = predicted_regimes == regime
    regime_returns = features.loc[mask, 'log_return']
    
    axes[i].hist(regime_returns, bins=30, alpha=0.7, density=True, 
                color=regime_colors.get(regime, 'blue'))
    axes[i].axvline(regime_returns.mean(), color='red', linestyle='--', 
                   label=f'Mean: {regime_returns.mean():.4f}')
    axes[i].set_title(f'{regime} Regime Returns', fontweight='bold')
    axes[i].set_xlabel('Log Return')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Next-Step Probability Prediction

Demonstrate the system's ability to predict next-step regime probabilities.

In [ ]:
# Demonstrate next-step probability computation
current_date = features.index[-1]
current_posterior = posterior_probs[-1]
current_state = predicted_states[-1]

print(f"Current Analysis (Date: {current_date.date()})")
print("=" * 60)

# Current regime probabilities
print("Current Regime Probabilities:")
for state_idx, prob in enumerate(current_posterior):
    regime_label = state_labels[state_idx]
    print(f"  {regime_label}: {prob:.3f}")

print(f"\nMost Likely Current Regime: {state_labels[current_state]}")

# Next-step probabilities
next_probs_from_posterior = best_model.compute_next_step_probabilities(
    current_posterior=current_posterior
)
next_probs_from_state = best_model.compute_next_step_probabilities(
    current_state=current_state
)

print("\nNext-Step Regime Probabilities:")
print("From Posterior Distribution:")
for i, prob in enumerate(next_probs_from_posterior):
    regime_label = state_labels[i]
    print(f"  {regime_label}: {prob:.3f}")

print("\nFrom Most Likely State:")
for i, prob in enumerate(next_probs_from_state):
    regime_label = state_labels[i]
    print(f"  {regime_label}: {prob:.3f}")

# Visualize next-step probabilities over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Plot recent price action
recent_days = 100
recent_idx = max(0, len(features) - recent_days)
recent_features = features.iloc[recent_idx:]
recent_prices = ohlc_data.loc[recent_features.index, 'Close']
recent_regimes = predicted_regimes.iloc[recent_idx:]

ax1.plot(recent_prices.index, recent_prices, 'k-', linewidth=1.5)
for regime in regime_colors:
    mask = recent_regimes == regime
    if mask.any():
        ax1.fill_between(recent_features.index, recent_prices.min(), recent_prices.max(),
                        where=mask, alpha=0.2, color=regime_colors[regime])

ax1.set_title('Recent Price Action with Detected Regimes', fontweight='bold')
ax1.set_ylabel('Price')
ax1.grid(True, alpha=0.3)

# Compute and plot next-step probabilities over time
next_step_history = []
for i in range(len(posterior_probs)):
    next_probs = posterior_probs[i] @ A
    next_step_history.append(next_probs)

next_step_history = np.array(next_step_history)

for state_idx, label in state_labels.items():
    ax2.plot(features.index[recent_idx:], 
            next_step_history[recent_idx:, state_idx],
            label=f'Next P({label})', linewidth=2, alpha=0.8)

ax2.set_title('Next-Step Regime Probabilities Over Time', fontweight='bold')
ax2.set_ylabel('Next-Step Probability')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 7. Simple Backtesting Demo

Let's run a simple backtest to see how regime-based strategies might perform.

In [ ]:
# Create a simple regime-based strategy
strategy = SimpleRegimeStrategy(allow_short=True)

# Generate signals
signals = strategy.generate_signals(
    predicted_regimes.tolist(),
    features,
    Close=ohlc_data.loc[features.index, 'Close']
)

# Simple portfolio simulation
returns = features['log_return'].shift(-1)  # Next day return
strategy_returns = signals * returns
strategy_returns = strategy_returns.dropna()

# Buy and hold benchmark
benchmark_returns = returns.dropna()

# Calculate performance metrics
strategy_perf = calculate_performance_metrics(strategy_returns)
benchmark_perf = calculate_performance_metrics(benchmark_returns)

print("Backtesting Results")
print("=" * 50)
print(f"{'Metric':<20} {'Strategy':<12} {'Buy & Hold':<12} {'Difference':<12}")
print("-" * 56)

metrics = ['total_return', 'cagr', 'annual_volatility', 'sharpe_ratio', 'max_drawdown']
for metric in metrics:
    strat_val = strategy_perf[metric]
    bench_val = benchmark_perf[metric]
    diff = strat_val - bench_val
    
    print(f"{metric.replace('_', ' ').title():<20} {strat_val:<12.4f} {bench_val:<12.4f} {diff:<+12.4f}")

# Plot cumulative performance
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Cumulative returns
strategy_cum_returns = (1 + strategy_returns).cumprod()
benchmark_cum_returns = (1 + benchmark_returns).cumprod()

ax1.plot(strategy_cum_returns.index, strategy_cum_returns, 
         label='Regime Strategy', linewidth=2, color='blue')
ax1.plot(benchmark_cum_returns.index, benchmark_cum_returns, 
         label='Buy & Hold', linewidth=2, color='gray', alpha=0.7)

ax1.set_title('Cumulative Returns Comparison', fontweight='bold')
ax1.set_ylabel('Cumulative Return')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Strategy positions
ax2.plot(signals.index, signals, alpha=0.8, linewidth=1, color='red')
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax2.fill_between(signals.index, 0, signals, alpha=0.3, color='red')
ax2.set_title('Strategy Positions', fontweight='bold')
ax2.set_ylabel('Position')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStrategy Statistics:")
print(f"Number of position changes: {(signals.diff() != 0).sum()}")
print(f"Average position: {signals.mean():.3f}")
print(f"Time in market: {(signals != 0).mean():.1%}")

## 8. Summary and Conclusions

This notebook demonstrated the complete HMM-based market regime detection workflow:

1. **Data Generation**: Created realistic market data with known regime characteristics
2. **Feature Engineering**: Computed 5 technical indicators (returns, volatility, RSI, MACD, Bollinger Width)
3. **Model Training**: Used model selection to find optimal number of states and trained HMM
4. **Regime Detection**: Applied Viterbi algorithm and forward-backward for regime identification
5. **Analysis**: Characterized detected regimes and compared with ground truth
6. **Prediction**: Demonstrated next-step probability computation
7. **Backtesting**: Showed simple regime-based trading strategy implementation

### Key Insights:
- HMM successfully identifies distinct market regimes with different return/volatility characteristics
- Transition probabilities provide valuable information about regime persistence
- Next-step probabilities enable forward-looking regime analysis
- Simple regime-based strategies can potentially outperform buy-and-hold in certain market conditions

### Next Steps:
1. Apply to real market data using the ingestion module
2. Implement more sophisticated trading strategies
3. Use rolling training for walk-forward analysis
4. Optimize strategy parameters and risk management
5. Deploy inference engine for real-time regime monitoring

In [ ]:
# Save model and results for later use
import joblib
from pathlib import Path

# Create output directory
output_dir = Path('../models/demo')
output_dir.mkdir(parents=True, exist_ok=True)

# Save model artifact
model_artifact = {
    'model': best_model,
    'scaler': scaler,
    'metadata': {
        'symbol': 'DEMO_DATA',
        'n_states': best_model.n_states,
        'feature_names': list(features.columns),
        'state_labels': state_labels,
        'training_samples': len(train_features),
        'model_selection_results': selection_results,
        'created_date': datetime.now().isoformat()
    }
}

artifact_path = output_dir / 'demo_hmm_model.joblib'
joblib.dump(model_artifact, artifact_path)

# Save processed data
demo_data = {
    'ohlc_data': ohlc_data,
    'features': features,
    'true_regimes': true_regimes,
    'predicted_regimes': predicted_regimes,
    'posterior_probabilities': posterior_probs,
    'strategy_signals': signals,
    'performance_results': {
        'strategy': strategy_perf,
        'benchmark': benchmark_perf
    }
}

data_path = output_dir / 'demo_data.joblib'
joblib.dump(demo_data, data_path)

print(f"Demo results saved to:")
print(f"  Model: {artifact_path}")
print(f"  Data: {data_path}")
print(f"\nYou can load these later for further analysis or to test the inference engine.")